# Advanced LearnM8 Configuration (Run Mode)

Learn how to customize learners and design sophisticated multi-stage workflows.

**What you'll learn:**
- Configure custom learners (Chemprop with custom parameters)
- Design multi-stage cycles (explore → exploit)
- Compare acquisition strategies (UCB, EI, Thompson, Greedy)
- Analyze selection quality trends (not enrichment)
- Visualize best score discovery progress

**Key Topics:**
1. **Custom Learners**: Chemprop MPNN with custom depth, message passing, FFN layers
2. **Custom Cycles**: Multi-stage workflows with different strategies per cycle
3. **Strategy Comparison**: Evaluate different acquisition functions

**Key Note:**
- **Run Mode**: Custom oracle generates scores on-the-fly
- **Metrics**: Selection quality metrics (avg score selected, not enrichment)
- **Focus**: Advanced configuration and workflow optimization

**Time estimate:** 15 minutes

## Section 1: Custom Learner Configuration (Chemprop)

In [ ]:
from learnm8 import run_active_learning
from learnm8.learners.torch.chemprop_learner import ChempropLearner
from examples.oracles import SimilarityOracle
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset (1K compounds, ID + SMILES only)
compounds = pl.read_csv('../data/ampc_1k_no_scores.csv')
print(f"Loaded {len(compounds)} compounds for custom learner experiment")

# Create oracle for similarity-based screening
oracle = SimilarityOracle(
    reference_smiles='c1ccc(cc1)C(=O)O',  # Benzoic acid scaffold
    fingerprint_type='morgan',
    metric='tanimoto'
)

print("\n=== Experiment 1: Custom Chemprop Learner ===")
print("Configuring Chemprop MPNN with custom parameters:")
print("  - depth=5 (default=3): More message passing layers")
print("  - message_hidden_dim=500 (default=300): Larger message vectors")
print("  - ffn_num_layers=3 (default=1): Deeper FFN")
print("  - batch_norm=True (default=False): Batch normalization")
print("  - dropout=0.1 (default=0.0): Regularization")

# Create custom Chemprop learner
custom_chemprop = ChempropLearner(
    depth=5,                    # Number of message passing steps
    message_hidden_dim=500,     # Hidden dimension of messages
    ffn_num_layers=3,           # Number of FFN layers
    ffn_hidden_dim=500,         # FFN hidden dimension
    batch_norm=True,            # Enable batch normalization
    dropout=0.1,                # Dropout probability
    max_epochs=30,              # Training epochs
    learning_rate=1e-4,         # Learning rate
    batch_size=32,              # Batch size
    random_state=42             # Reproducibility
)

# Run with custom Chemprop learner
# Note: Chemprop works directly with SMILES (no featurizer needed)
chemprop_results = run_active_learning(
    compound_pool=compounds,           # DataFrame with ID and SMILES
    oracle=oracle,                     # SimilarityOracle
    target_col='similarity',           # Property name
    learner=custom_chemprop,           # Custom Chemprop instance
    # featurizer_type NOT specified - Chemprop uses SMILES directly
    strategy='greedy',                 # Greedy selection
    n_cycles=5,                        # 5 cycles
    batch_fraction=0.02,               # 2% per cycle
    score_direction='higher'           # Higher similarity is better
)

print(f"\n✓ Custom Chemprop experiment complete!")
print(f"  Compounds measured: {len(chemprop_results['measured_ids'])}")

# Analyze selection quality
measured_df = chemprop_results['compounds_df'].filter(pl.col('measured'))
print(f"  Average similarity (selected): {measured_df['similarity'].mean():.3f}")
print(f"  Top similarity found: {measured_df['similarity'].max():.3f}")

# Plot Chemprop performance
chemprop_metrics = pd.DataFrame(chemprop_results['cycle_metrics'])

# Calculate best similarity found so far per cycle
chemprop_measured_df = chemprop_results['compounds_df'].filter(pl.col('measured'))
chemprop_best_scores = []
for cycle_num in range(len(chemprop_metrics)):
    cycle_measured = chemprop_measured_df.filter(pl.col('cycle_measured') <= cycle_num)
    chemprop_best_scores.append(cycle_measured['similarity'].max())

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(chemprop_metrics['cycle'], chemprop_metrics['avg_score_selected'],
         marker='o', linewidth=2, color='purple')
plt.xlabel('Cycle')
plt.ylabel('Avg Similarity (Selected)')
plt.title('Chemprop Selection Quality')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(chemprop_metrics['cycle'], chemprop_best_scores,
         marker='s', linewidth=2, color='green')
plt.xlabel('Cycle')
plt.ylabel('Best Similarity Found')
plt.title('Best Score Discovery Progress')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(chemprop_metrics['cycle'], chemprop_metrics['n_measured'].cumsum(),
         marker='^', linewidth=2, color='orange')
plt.xlabel('Cycle')
plt.ylabel('Cumulative Measurements')
plt.title('Measurement Budget')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Chemprop Benefits:")
print("  - Works directly with SMILES (no featurizer needed)")
print("  - Learns from molecular graph structure")
print("  - Customizable architecture (depth, hidden dims, dropout)")
print("  - Good for small-to-medium datasets")

## Section 2: Custom Multi-Stage Cycles

In [ ]:
from learnm8 import CycleConfig

print("\n=== Experiment 2: Custom Multi-Stage Cycles ===")
print("Demonstrating three approaches to cycle configuration:\n")
print("1. Simple list: Random → Greedy (exploitation-focused)")
print("2. CycleConfig list: UCB with varying beta (exploration → exploitation)")
print("3. Mixed approach: CycleConfig + simple tuples\n")

# Strategy 1: Simple list definition (random → greedy)
print("\nRunning Strategy 1: Simple List (Random → Greedy)...")
print("  Approach: Pure exploitation after initialization")
results_simple = run_active_learning(
    compound_pool=compounds,
    oracle=oracle,
    learner='rf',                      # Random Forest (no uncertainty needed)
    target_col='similarity',
    featurizer_type='morgan',
    cycles=[
        ('random', 0.02),      # 2% initialization
        ('greedy', 0.01),      # 1% greedy x4
        ('greedy', 0.01),
        ('greedy', 0.01),
        ('greedy', 0.01)
    ],
    score_direction='higher'  # Higher similarity is better
)

# Strategy 2: CycleConfig list with custom UCB parameters
print("\nRunning Strategy 2: CycleConfig List (UCB with Varying Beta)...")
print("  Approach: Start with high exploration (beta=3.0), gradually reduce to exploitation (beta=0.5)")
results_config = run_active_learning(
    compound_pool=compounds,
    oracle=oracle,
    learner='rf_ensemble',             # RF Ensemble for UCB uncertainty
    target_col='similarity',
    featurizer_type='morgan',
    cycles=[
        CycleConfig('random', n_cycles=1, batch_fraction=0.02),
        CycleConfig('ucb', n_cycles=1, batch_fraction=0.01,
                    acquisition_params={'beta': 10.0}),  # High exploration
        CycleConfig('ucb', n_cycles=1, batch_fraction=0.01,
                    acquisition_params={'beta': 5.0}),  # Balanced
        CycleConfig('ucb', n_cycles=1, batch_fraction=0.01,
                    acquisition_params={'beta': 2.0}),  # Moderate exploitation
        CycleConfig('ucb', n_cycles=1, batch_fraction=0.01,
                    acquisition_params={'beta': 1.0})   # Strong exploitation
    ],
    score_direction='higher'
)

# Strategy 3: Mixed approach (CycleConfig + simple tuples)
print("\nRunning Strategy 3: Mixed Approach (CycleConfig + Tuples)...")
print("  Approach: Initialization → UCB with custom beta → Greedy refinement")
results_mixed = run_active_learning(
    compound_pool=compounds,
    oracle=oracle,
    learner='rf_ensemble',             # RF Ensemble for UCB
    target_col='similarity',
    featurizer_type='morgan',
    cycles=[
        ('random', 0.02),              # Simple tuple: initialization
        CycleConfig('ucb', n_cycles=2, batch_fraction=0.01,
                    acquisition_params={'beta': 2.5}),  # CycleConfig: exploration
        CycleConfig('ucb', n_cycles=1, batch_fraction=0.01,
                    acquisition_params={'beta': 1.0}),  # CycleConfig: exploitation
        ('greedy', 0.01)               # Simple tuple: final refinement
    ],
    score_direction='higher'
)

print("\n✓ All multi-stage strategies complete")

# Plot comparison (RUN MODE: avg score selected and best score found)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

strategies = [
    (results_simple, 'Simple List\n(Random → Greedy)'),
    (results_config, 'CycleConfig\n(UCB β=3.0→0.5)'),
    (results_mixed, 'Mixed\n(Tuples + CycleConfig)')
]

for idx, (result, name) in enumerate(strategies):
    metrics = pd.DataFrame(result['cycle_metrics'])
    measured_df = result['compounds_df'].filter(pl.col('measured'))

    # Calculate best score found so far per cycle
    best_scores = []
    for cycle_num in range(len(metrics)):
        cycle_measured = measured_df.filter(pl.col('cycle_measured') <= cycle_num)
        best_scores.append(cycle_measured['similarity'].max())

    # Top row: Average score selected
    axes[0, idx].plot(metrics['cycle'], metrics['avg_score_selected'],
                      marker='o', linewidth=2, color='blue')
    axes[0, idx].set_title(name)
    axes[0, idx].set_xlabel('Cycle')
    axes[0, idx].set_ylabel('Avg Similarity (Selected)')
    axes[0, idx].grid(True, alpha=0.3)

    # Bottom row: Best score discovered
    axes[1, idx].plot(metrics['cycle'], best_scores,
                      marker='s', linewidth=2, color='green')
    axes[1, idx].set_xlabel('Cycle')
    axes[1, idx].set_ylabel('Best Similarity Found')
    axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics (RUN MODE: selection metrics only, not enrichment)
print("\nMulti-Stage Strategy Comparison:")
for result, name in strategies:
    metrics = pd.DataFrame(result['cycle_metrics'])
    measured_df = result['compounds_df'].filter(pl.col('measured'))
    avg_score = measured_df['similarity'].mean()
    top_score = measured_df['similarity'].max()
    total_measured = len(result['measured_ids'])
    print(f"\n{name.replace(chr(10), ' ')}:")  # Remove newlines for print
    print(f"  Average similarity (selected): {avg_score:.3f}")
    print(f"  Best similarity found: {top_score:.3f}")
    print(f"  Total measured: {total_measured}")

print("\n💡 Multi-Stage Cycle Configuration Approaches:")
print("  1. Simple List: Easy to use, good for basic workflows (tuples)")
print("  2. CycleConfig List: Full control over acquisition parameters (e.g., beta tuning)")
print("  3. Mixed: Combine simplicity and control where needed")
print("\n💡 Strategy Insights:")
print("  - Simple List: Pure exploitation maximizes immediate score gains")
print("  - CycleConfig: Gradually reducing beta balances exploration → exploitation")
print("  - Mixed: Flexible approach adapts to workflow requirements")

## Key Insights

**Advanced Configuration Topics:**
1. **Custom Learners**: Chemprop MPNN with custom depth, message passing, FFN layers
2. **Cycle Configuration**: Three approaches (simple lists, CycleConfig with custom params, mixed)
3. **Multi-Stage Workflows**: Design exploration → exploitation transitions
4. **Strategy Comparison**: Evaluate greedy, UCB, EI, Thompson acquisition functions

**Run Mode Metrics:**
- ✅ Average score of selected compounds per cycle
- ✅ Top score found overall
- ✅ Cumulative measurements
- ✅ Selection quality trends
- ❌ Enrichment factor (requires ground truth)
- ❌ Top-K discovery (requires ground truth)

**Custom Learner Configuration:**
- **Chemprop**: Works directly with SMILES (no featurizer)
- Customizable: depth, message_hidden_dim, ffn_num_layers, dropout, batch_norm
- Good for: Small-to-medium datasets, learning from graph structure
- Pass custom learner instance to `run_active_learning(learner=custom_learner)`

**Cycle Configuration Methods:**
1. **Simple Lists:** Use tuples `[('random', 0.02), ('greedy', 0.01), ...]` for straightforward workflows
2. **CycleConfig Objects:** Use `CycleConfig('ucb', n_cycles=1, acquisition_params={'beta': 2.0})` for fine-grained control
3. **Mixed Approach:** Combine both for flexibility (tuples where simple, CycleConfig where custom params needed)

**Multi-Stage Workflow Design:**
1. **Initialization:** Random sampling establishes baseline (1-2%)
2. **Exploration → Exploitation:** Use CycleConfig to gradually adjust parameters (e.g., UCB beta=10.0 → 1.0)
3. **Refinement:** Final greedy cycles to maximize top candidates

**Strategy Selection:**
- **Pure Exploitation:** Simple list with random → greedy (maximizes immediate gains)
- **Adaptive Exploration:** CycleConfig with varying beta (balances exploration → exploitation)
- **Flexible:** Mixed approach adapts to specific workflow needs

**Learner Selection:**
- Use `rf_ensemble` for uncertainty-based strategies (UCB, EI, Thompson)
- Use `rf` for greedy or diversity-based strategies
- Use custom learner instances for specialized architectures (Chemprop, etc.)

**Next:** [04_production_screening.ipynb](04_production_screening.ipynb) for real-world deployment